由于本地是 TensorFlow 2.x，运行时会产生兼容性问题，故在保证功能不变的基础上对原有代码进行部分注释和改动。

In [9]:
import tensorflow as tf
import numpy as np

# 原代码（TensorFlow 2.x 已不可用）
# from tensorflow.examples.tutorials.mnist import input_data
# mnist = input_data.read_data_sets('MNIST_data', one_hot=True)

# TensorFlow 2.x 下启用 1.x 图模式
tf.compat.v1.disable_eager_execution()

# 使用 Keras 自带 MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1.0})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1.0})
    return result

def weight_variable(shape):
    # 原代码
    # initial = tf.truncated_normal(shape, stddev=0.1)
    initial = tf.random.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 原代码应补 tf.nn.conv2d，但为兼容 TF2，直接写 compat.v1
    return tf.compat.v1.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool2d(x, ksize=2, strides=2, padding='SAME') 

# define placeholder for inputs to network
# 原代码
# xs = tf.placeholder(tf.float32, [None, 784])/255.
# ys = tf.placeholder(tf.float32, [None, 10])
# keep_prob = tf.placeholder(tf.float32)

xs = tf.compat.v1.placeholder(tf.float32, [None, 784]) / 255.
ys = tf.compat.v1.placeholder(tf.float32, [None, 10])
keep_prob = tf.compat.v1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 卷积层 1
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层 1
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)

h_fc1_drop = tf.nn.dropout(h_fc1, rate=1 - keep_prob)

# 全连接层 2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])

# 原代码
# prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

logits = tf.matmul(h_fc1_drop, W_fc2) + b_fc2
prediction = tf.nn.softmax(logits)

# 交叉熵函数
# 原代码
# cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction),
#                                               reduction_indices=[1]))

cross_entropy = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits(labels=ys, logits=logits)
)

# 原代码
# train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

# 原代码
# with tf.Session() as sess:
#     init = tf.global_variables_initializer()

with tf.compat.v1.Session() as sess:
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)

    for i in range(max_epoch):
        # 原代码（TF2 下 mnist.train.next_batch 不存在）
        # batch_xs, batch_ys = mnist.train.next_batch(100)

        idx = np.random.choice(len(x_train), 100, replace=False)
        batch_xs = x_train[idx].reshape(-1, 784).astype(np.float32)
        batch_ys = y_train[idx]

        sess.run(train_step, feed_dict={
            xs: batch_xs,
            ys: batch_ys,
            keep_prob: keep_prob_rate
        })

        if i % 100 == 0:
            test_images = x_test[:1000].reshape(-1, 784).astype(np.float32)
            test_labels = y_test[:1000]
            print(compute_accuracy(test_images, test_labels))

0.069
0.859
0.893
0.921
0.928
0.936
0.942
0.951
0.951
0.954
0.957
0.953
0.956
0.957
0.958
0.961
0.967
0.963
0.966
0.972
